# Calibration des seuils de l'état de la pièce

Le modèle v2 donne une probabilité « endommagé ». L'application en déduit trois états :
**bon état** (sous le seuil bas), **endommagé** (au-dessus du seuil haut) et **incertain** entre les deux.
Une pièce jugée saine n'accuse aucun composant; un état incertain réduit le poids de l'image dans la fusion.

Les seuils sont choisis sur le **jeu de validation** (`data/v2/val`), jamais sur le test.
Les photos réelles servent seulement de vérification.

Règle fixée d'avance : on retient la zone incertaine **la plus étroite** pour laquelle les images hors zone sont
classées correctement à au moins 97 %.

Exécuter après `entrainement_v2.ipynb`. Cliquez sur **Run All** (environ 2 minutes).

In [ ]:
import json
import sys
from pathlib import Path

RACINE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RACINE))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from diagnostic import fusion
from diagnostic.vision import DetecteurEtat

EXACTITUDE_VISEE = 0.97
CANDIDATS = [(0.5, 0.5), (0.4, 0.6), (0.3, 0.7), (0.2, 0.8), (0.1, 0.9)]
detecteur = DetecteurEtat()
plt.rcParams["figure.dpi"] = 110

def probabilites(dossier, classe_de):
    """Probabilité « endommagé » de chaque image d'un dossier; classe_de(chemin) donne la vraie classe."""
    lignes = []
    for chemin in sorted(dossier.rglob("*")):
        if chemin.suffix.lower() in {".jpg", ".jpeg", ".png"}:
            lignes.append({"fichier": chemin.name, "classe": classe_de(chemin),
                           "p_endommage": detecteur.proba_endommage(chemin)})
    return pd.DataFrame(lignes)

## 1. Probabilités sur le jeu de validation

In [ ]:
val = probabilites(RACINE / "data" / "v2" / "val", lambda chemin: chemin.parent.name)
val["source"] = val["fichier"].str.split("_").str[0]

fig, ax = plt.subplots(figsize=(9, 3.6))
for classe, couleur in [("bon_etat", "#2f9e44"), ("endommage", "#c92a2a")]:
    ax.hist(val.loc[val["classe"] == classe, "p_endommage"], bins=40, range=(0, 1), alpha=0.6,
            color=couleur, label=classe.replace("_", " "))
ax.set(yscale="log", xlabel="Probabilité « endommagé »", ylabel="Images (échelle log)",
       title=f"Validation : {len(val)} images")
ax.legend()
plt.tight_layout()
plt.show()

## 2. Comparaison des seuils

Pour chaque paire (seuil bas, seuil haut) : la part des images classées « incertain », et l'exactitude
sur les images hors de cette zone.

In [ ]:
def mesurer(df, bas, haut):
    endommage = df["classe"] == "endommage"
    hors_zone = (df["p_endommage"] < bas) | (df["p_endommage"] >= haut)
    predit = df["p_endommage"] >= haut
    return {"Seuil bas": bas, "Seuil haut": haut,
            "Incertain": 1 - hors_zone.mean(),
            "Exactitude hors zone": (predit == endommage)[hors_zone].mean(),
            "Endommagées jugées saines": (endommage & (df["p_endommage"] < bas)).sum(),
            "Saines jugées endommagées": (~endommage & predit).sum()}

tableau = pd.DataFrame([mesurer(val, bas, haut) for bas, haut in CANDIDATS])
retenus = tableau[tableau["Exactitude hors zone"] >= EXACTITUDE_VISEE]
choix = (retenus if len(retenus) else tableau.sort_values("Exactitude hors zone", ascending=False)).iloc[0]
print(f"Seuils retenus : bon état < {choix['Seuil bas']}, endommagé ≥ {choix['Seuil haut']}")
print(f"Valeurs actuelles dans diagnostic/fusion.py : SEUIL_BON_ETAT = {fusion.SEUIL_BON_ETAT}, "
      f"SEUIL_ENDOMMAGE = {fusion.SEUIL_ENDOMMAGE}")
tableau.style.format({"Incertain": "{:.1%}", "Exactitude hors zone": "{:.1%}"}).hide(axis="index")

In [ ]:
# Détail par source avec les seuils retenus
pd.DataFrame({s: mesurer(g, choix["Seuil bas"], choix["Seuil haut"]) for s, g in val.groupby("source")}).T \
    .drop(columns=["Seuil bas", "Seuil haut"]).style.format({"Incertain": "{:.1%}", "Exactitude hors zone": "{:.1%}"})

## 3. Vérification sur les photos réelles

Les photos de `data/photos_reelles/` (dossier `aucun` = bon état). Elles ne servent **pas** à choisir les seuils.

In [ ]:
reelles = probabilites(RACINE / "data" / "photos_reelles",
                      lambda chemin: "bon_etat" if chemin.parent.name == "aucun" else "endommage")
reelles["état"] = reelles["p_endommage"].map(
    lambda p: "bon état" if p < choix["Seuil bas"] else ("endommagé" if p >= choix["Seuil haut"] else "incertain"))
print(pd.Series(mesurer(reelles, choix["Seuil bas"], choix["Seuil haut"])).to_string())
reelles.style.format({"p_endommage": "{:.0%}"}).hide(axis="index")

In [ ]:
(RACINE / "resultats").mkdir(exist_ok=True)
_ = (RACINE / "resultats" / "calibration_seuil.json").write_text(json.dumps({
    "seuil_bon_etat": float(choix["Seuil bas"]), "seuil_endommage": float(choix["Seuil haut"]),
    "validation": json.loads(tableau.to_json(orient="records")),
}, indent=2), encoding="utf-8")
print("Si les seuils retenus diffèrent des valeurs actuelles, modifiez SEUIL_BON_ETAT et SEUIL_ENDOMMAGE",
      "dans diagnostic/fusion.py, puis relancez pytest.")